# Load DWH – CSV files → PostgreSQL star schema

Loads the files written by `Ingest_data.ipynb` into the DWH (design: `docs/DWH_DESIGN.md`).

| CSV | Staging | Dimension / fact | Procedure |
|---|---|---|---|
| `config/weather_types.csv` | `stg.weather_types` | `dim_weather_type` | `load_dim_weather_type` – only when codes are missing |
| `raw_data/weather/deltadate=*/<City>.csv` | `stg.weather_daily` | `dim_date` | `load_dim_date` – only when dates are missing |
| `raw_data/cities/deltadate=*/cities.csv` | `stg.cities` | `dim_city` | `load_dim_city` (SCD1, newest deltadate per city) |
| `raw_data/weather/deltadate=*/<City>.csv` | `stg.weather_daily` | `fact_weather_daily` | `load_fact_weather_daily` |

Only weather files that are not in the fact yet (deltadate + city) are loaded, so re-running is cheap and a new
city gets its whole history. Everything runs in one transaction: a failed load changes nothing.
The database is `DATABASE_URL` in `.env` (see `.env.example`); raw files are read from `raw_data/` or the bucket (`STORAGE`).

## Setup

In [ ]:
import io
import sys
from pathlib import Path

import boto3
import pandas as pd
import psycopg
from botocore.config import Config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from config.settings import (DATABASE_URL, RAW_DIR, S3_ACCESS_KEY_ID, S3_BUCKET, S3_ENDPOINT_URL, S3_REGION,
                             S3_SECRET_ACCESS_KEY, STORAGE)

raw = ROOT / RAW_DIR
if STORAGE == "s3":
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT_URL, region_name=S3_REGION, aws_access_key_id=S3_ACCESS_KEY_ID,
                      aws_secret_access_key=S3_SECRET_ACCESS_KEY, config=Config(s3={"addressing_style": "path"}))


def list_keys(prefix):
    """Keys of the raw CSV files under prefix, e.g. 'weather/' -> ['weather/deltadate=2026-10-06/Atlanta.csv', ...]"""
    if STORAGE == "s3":
        pages = s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=prefix)
        return sorted(o["Key"] for page in pages for o in page.get("Contents", []) if o["Key"].endswith(".csv"))
    return sorted(p.relative_to(raw).as_posix() for p in (raw / prefix).rglob("*.csv"))


def read_csv(key):
    if STORAGE == "s3":
        return pd.read_csv(s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"])
    return pd.read_csv(raw / key)


def deltadate_of(key):
    return key.split("/")[1].split("=")[1]          # weather/deltadate=2026-10-06/Atlanta.csv -> 2026-10-06


conn = psycopg.connect(DATABASE_URL)
conn.add_notice_handler(lambda d: None if "already exists" in d.message_primary else print(d.message_primary))
print(f"raw files: {STORAGE} | DWH: {conn.info.host}:{conn.info.port}/{conn.info.dbname}")

## Tables and procedures
`sql/*.sql` – safe to run every time.

In [ ]:
for script in sorted((ROOT / "sql").glob("*.sql")):
    conn.execute(script.read_text(encoding="utf-8"))
    print(script.name)
conn.commit()

## Functions
* `query` – SQL result as a DataFrame
* `stage` – empties a staging table and copies a DataFrame into it (columns by name, extra CSV columns are ignored)

In [ ]:
def query(sql):
    cur = conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def stage(table, df):
    columns = query(f"""SELECT column_name FROM information_schema.columns
                         WHERE table_schema = 'stg' AND table_name = '{table}' ORDER BY ordinal_position""")["column_name"]
    data = io.StringIO()
    df.reindex(columns=columns).to_csv(data, index=False, header=False)
    conn.execute(f"TRUNCATE stg.{table}")
    with conn.cursor().copy(f"COPY stg.{table} ({', '.join(columns)}) FROM STDIN WITH (FORMAT csv)") as copy:
        copy.write(data.getvalue())
    print(f"stg.{table}: {len(df)} rows")

## Staging – CSV files → stg

In [ ]:
loaded = query("""SELECT DISTINCT f.deltadate::text AS deltadate, replace(c.city, ' ', '_') AS file
                   FROM dwh.fact_weather_daily f JOIN dwh.dim_city c USING (city_key)""")
loaded = set(zip(loaded["deltadate"], loaded["file"]))

new_files = [k for k in list_keys("weather/") if (deltadate_of(k), k.rsplit("/", 1)[1][:-4]) not in loaded]
print(f"{len(new_files)} new weather file(s)")

stage("weather_types", pd.read_csv(ROOT / "config" / "weather_types.csv"))
stage("cities", pd.concat([read_csv(k) for k in list_keys("cities/")]))
stage("weather_daily", pd.concat([read_csv(k) for k in new_files]) if new_files else pd.DataFrame())

## Dimensions – date and weather type, only when something is missing

In [ ]:
missing_types = query("""SELECT count(*) AS n FROM stg.weather_types s
                          WHERE NOT EXISTS (SELECT 1 FROM dwh.dim_weather_type d
                                            WHERE d.weather_type_key = s.weather_code)""")["n"][0]
if missing_types:
    conn.execute("CALL dwh.load_dim_weather_type()")
else:
    print("dim_weather_type: all codes exist")

missing_dates = query("""SELECT count(DISTINCT s.date) AS n FROM stg.weather_daily s
                          WHERE NOT EXISTS (SELECT 1 FROM dwh.dim_date d WHERE d.date = s.date)""")["n"][0]
if missing_dates:
    conn.execute("CALL dwh.load_dim_date()")
else:
    print("dim_date: all dates exist")

## City dimension and fact

In [ ]:
conn.execute("CALL dwh.load_dim_city()")
conn.execute("CALL dwh.load_fact_weather_daily()")
conn.commit()

## Check

In [ ]:
query("""
SELECT 'dim_date' AS "table", count(*) AS rows, min(date)::text AS "from", max(date)::text AS "to" FROM dwh.dim_date
UNION ALL SELECT 'dim_weather_type', count(*), NULL, NULL FROM dwh.dim_weather_type
UNION ALL SELECT 'dim_city', count(*), NULL, NULL FROM dwh.dim_city
UNION ALL SELECT 'fact_weather_daily', count(*), min(deltadate)::text, max(deltadate)::text FROM dwh.fact_weather_daily
""")

In [ ]:
query("""
SELECT c.city, c.country, count(*) AS days, round(avg(f.temperature_2m_mean)::numeric, 1) AS avg_temp_c,
       round(sum(f.precipitation_sum)::numeric) AS precipitation_mm,
       count(*) FILTER (WHERE wt.category IN ('Rain', 'Drizzle', 'Storm')) AS rainy_days
FROM dwh.fact_weather_daily f
JOIN dwh.dim_city c USING (city_key)
JOIN dwh.dim_weather_type wt USING (weather_type_key)
GROUP BY 1, 2 ORDER BY avg_temp_c DESC
""")